In [1]:
print("ok all")

ok all


### 1. Environment Setup

In [2]:
# Standard library imports
import os
import json
import time
from typing import List, Dict, Any, Optional
from datetime import datetime

# Third-party imports
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Haystack imports
from haystack import Document, Pipeline
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.generators.azure import AzureOpenAIGenerator
from haystack.components.builders import PromptBuilder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.dataclasses import ChatMessage
from haystack.utils import Secret


import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful!")

✅ All imports successful!


### 2. Understanding RAG Components

#### 2.1 Document Representation

In [3]:
doc = Document(
    content="Haystack is an open-source framework for building LLM applications.",
    meta={
        "source": "documentation",
        "category": "introduction",
        "date": "2025-01-01"        
    }
)

print("Document ID:", doc.id)
print("Content:", doc.content)
print("Metadata:", doc.meta)

Document ID: 765db240323e56967d8da988e047efcdc3610cf8615e8f31dfb3eae69e1dea2a
Content: Haystack is an open-source framework for building LLM applications.
Metadata: {'source': 'documentation', 'category': 'introduction', 'date': '2025-01-01'}


#### 2.2 Document Store
The Document Store is where all documents are stored. We'll use InMemoryDocumentStore for simplicity.

In [4]:
document_store = InMemoryDocumentStore()

sample_docs = [
    Document(content="Python is a high-level programming language."),
    Document(content="Machine learning is a subset of artificial intelligence."),
    Document(content="RAG combines retrieval with generation for better LLM responses.")    
]

document_store.write_documents(sample_docs)

print(f"✅ Stored {document_store.count_documents()} documents")
print("\nDocuments in store:")
for doc in document_store.filter_documents():
    print(f"  - {doc.content[:50]}...")

✅ Stored 3 documents

Documents in store:
  - Python is a high-level programming language....
  - Machine learning is a subset of artificial intelli...
  - RAG combines retrieval with generation for better ...


#### 2.3 Retriever
The Retriever finds relevant documents based on a query. We'll use BM25 (keyword-based) retrieval.

In [5]:
retriever = InMemoryBM25Retriever(document_store=document_store)

query = "What is RAG?"
result = retriever.run(query=query)

print(f"Retrieved {len(result['documents'])} documents:\n")
for i, doc in enumerate(result['documents'], 1):
    print(f"{i}. {doc.content}")
    print(f"   Score: {doc.score:.4f}\n")

Retrieved 3 documents:

1. RAG combines retrieval with generation for better LLM responses.
   Score: 1.4837

2. Python is a high-level programming language.
   Score: 1.2201

3. Machine learning is a subset of artificial intelligence.
   Score: 1.2005



#### 2.4 Prompt Builder
The Prompt Builder constructs prompts by combining the query with retrieved documents.

In [6]:
template = """
Answer the question based on the context provided.

Context:
{% for doc in documents %}
  {{ doc.content }}
{% endfor %}

Question: {{ query }}

Answer:
"""
prompt_builder = PromptBuilder(template=template)
prompt_result = prompt_builder.run(
    documents=result['documents'],
    query=query
)

print("Generated Prompt:")
print("=" * 50)
print(prompt_result['prompt'])
print("=" * 50)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


Generated Prompt:

Answer the question based on the context provided.

Context:

  RAG combines retrieval with generation for better LLM responses.

  Python is a high-level programming language.

  Machine learning is a subset of artificial intelligence.


Question: What is RAG?

Answer:


#### 2.5 Generator
The Generator (LLM) produces the final answer based on the prompt.

In [7]:
generator = AzureOpenAIGenerator(
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version="2024-12-01-preview",
    generation_kwargs={"temperature": 0.3},
)
answer=generator.run(prompt=prompt_result['prompt'])

print("Generated Answer:")
print("=" * 50)
print(answer['replies'][0])
print("=" * 50)

Generated Answer:
RAG (Retrieval-Augmented Generation) is a method that combines retrieval of relevant information with generation to produce better responses from large language models (LLMs).


### 3. Data Preparation

In [8]:
def generate_company_knowledge_base()-> List[Document]:
    """
    Generate a synthetic company knowledge base with policies and information.
    
    Returns:
        List of Document objects representing company policies
    """
    company_data = [
        {
            "title": "Remote Work Policy",
            "content": """TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests must be approved by direct managers. Employees must be available 
            during core hours (10 AM - 3 PM local time) and maintain productivity standards. 
            Equipment for remote work is provided by the company.""",
            "category": "HR",
            "last_updated": "2025-01-15"
        },
        {
            "title": "Vacation Policy",
            "content": """Full-time employees receive 20 days of paid vacation annually. 
            Vacation days accrue monthly (1.67 days per month). Employees must submit vacation 
            requests at least 2 weeks in advance through the HR portal. Unused vacation days 
            can be carried over up to 5 days into the next year.""",
            "category": "HR",
            "last_updated": "2025-01-10"
        },
        {
            "title": "Health Insurance",
            "content": """TechCorp offers comprehensive health insurance including medical, 
            dental, and vision coverage. The company covers 80% of premium costs for employees 
            and 50% for dependents. Insurance becomes effective on the first day of the month 
            following 30 days of employment.""",
            "category": "Benefits",
            "last_updated": "2025-01-20"
        },
        {
            "title": "Professional Development",
            "content": """Employees have access to $2,000 annually for professional development, 
            including conferences, courses, and certifications. Requests must be submitted through 
            the learning management system and approved by managers. Employees must remain with 
            the company for 12 months after using these funds or repay the amount.""",
            "category": "Learning",
            "last_updated": "2025-01-05"
        },
        {
            "title": "Code of Conduct",
            "content": """All employees must adhere to TechCorp's code of conduct, which includes 
            treating colleagues with respect, maintaining confidentiality, avoiding conflicts of 
            interest, and reporting unethical behavior. Violations may result in disciplinary 
            action up to and including termination.""",
            "category": "Compliance",
            "last_updated": "2024-12-01"
        },
        {
            "title": "Equipment and Technology",
            "content": """TechCorp provides all necessary equipment including laptops, monitors, 
            and software licenses. Employees can choose between Mac or Windows laptops. 
            Equipment is refreshed every 3 years. Personal use of company equipment is permitted 
            within reasonable limits.""",
            "category": "IT",
            "last_updated": "2025-01-12"
        },
        {
            "title": "Performance Reviews",
            "content": """Performance reviews are conducted bi-annually in January and July. 
            Reviews include self-assessment, peer feedback, and manager evaluation. Performance 
            ratings directly impact compensation adjustments and promotion eligibility. 
            Employees receive written feedback and development plans.""",
            "category": "HR",
            "last_updated": "2025-01-08"
        },
        {
            "title": "Security Protocols",
            "content": """All employees must use two-factor authentication for company systems. 
            Passwords must be at least 12 characters with complexity requirements. Employees 
            must not share credentials or leave workstations unlocked. Security incidents 
            must be reported immediately to the IT security team.""",
            "category": "Security",
            "last_updated": "2025-01-18"
        }
    ]
    documents = []
    for item in company_data:
        doc = Document(
            content=item["content"],
            meta={
                "title": item["title"],
                "category": item["category"],
                "last_updated": item["last_updated"],
                "source": "company_kb"
            }
        )
        documents.append(doc)
    return documents

company_docs = generate_company_knowledge_base()
print(f"✅ Generated {len(company_docs)} company documents")
print("\nSample document:")
print(f"Title: {company_docs[0].meta['title']}")
print(f"Content: {company_docs[0].content[:100]}...")
    

✅ Generated 8 company documents

Sample document:
Title: Remote Work Policy
Content: TechCorp allows employees to work remotely up to 3 days per week. 
            Remote work requests ...


### 4. Example 1: Simple Q&A System

#### 4.1 Create Pipeline Components

In [9]:
company_store = InMemoryDocumentStore()
company_store.write_documents(company_docs)

print(f"📚 Document store contains {company_store.count_documents()} documents")

📚 Document store contains 8 documents


In [10]:
rag_template =  """
You are a helpful HR assistant for TechCorp. Answer the employee's question based on the company policies provided.

Company Policies:
{% for doc in documents %}
---
Policy: {{ doc.meta.title }}
{{ doc.content }}
{% endfor %}

Employee Question: {{ query }}

Instructions:
- Provide a clear, concise answer based on the policies above
- If the information isn't in the policies, say so honestly
- Cite the specific policy title when applicable
- Be friendly and professional

Answer:
"""

#### 4.2 Build the RAG Pipeline

In [11]:
retriever = InMemoryBM25Retriever(document_store=company_store)
prompt_builder = PromptBuilder(template=rag_template)
llm = AzureOpenAIGenerator(
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version="2024-12-01-preview",
    generation_kwargs={"temperature": 0.3},
)
rag_pipeline = Pipeline()
rag_pipeline.add_component("retriever",retriever)
rag_pipeline.add_component("prompt_builder",prompt_builder)
rag_pipeline.add_component("llm",llm)

rag_pipeline.connect("retriever", "prompt_builder.documents")
rag_pipeline.connect("prompt_builder","llm")


print("✅ RAG Pipeline created successfully!")

# Visualize the pipeline
print("\nPipeline structure:")
print(rag_pipeline)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ RAG Pipeline created successfully!

Pipeline structure:
🚅 Components
  - retriever: InMemoryBM25Retriever
  - prompt_builder: PromptBuilder
  - llm: AzureOpenAIGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (list[Document])
  - prompt_builder.prompt -> llm.prompt (str)



In [24]:
def ask_question(pipeline: Pipeline, question: str, top_k: int=3)-> Dict[str,Any]:
    """
    Ask a question using the RAG pipeline.
    """
    start_time = time.time()
    result = pipeline.run({
    "retriever": {"query": question, "top_k": top_k},
    "prompt_builder": {"query": question}
    },
    include_outputs_from = {"retriever", "prompt_builder", "llm"})
    end_time = time.time()
    print(f"DEBUG - Result keys: {result.keys()}")

    if "retriever" in result:
        retriever_docs = result["retriever"]["documents"]
    else:
        print("WARNING: No retriever output found!")
        retrieved_docs = []
    return {
        "question": question,
        "answer": result['llm']["replies"][0],
        "retrieved_docs": retriever_docs,
        "response_time": end_time-start_time
    }



In [15]:
test_questions = [
    "How many vacation days do I get per year?",
    "Can I work from home?",
    "What's the health insurance coverage?"
]
for question in test_questions:
    print("\n" + "="*80)
    print(f"❓ Question: {question}")
    print("="*80)
    
    result = ask_question(rag_pipeline, question)
    
    print(f"\n💡 Answer:\n{result['answer']}")
    print(f"\n⏱️  Response time: {result['response_time']:.2f}s")
    print(f"\n📄 Retrieved {len(result['retrieved_docs'])} relevant policies:")
    for i, doc in enumerate(result['retrieved_docs'], 1):
        print(f"   {i}. {doc.meta['title']} (score: {doc.score:.3f})")


❓ Question: How many vacation days do I get per year?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
Hello! Thank you for your question. 

According to TechCorp's **Vacation Policy**, full-time employees receive **20 days of paid vacation annually**. These vacation days accrue monthly at a rate of **1.67 days per month**. Additionally, unused vacation days can be carried over into the next year, up to a maximum of **5 days**.

If you have further questions or need assistance with submitting a vacation request, feel free to reach out. Have a great day! 😊

⏱️  Response time: 3.32s

📄 Retrieved 3 relevant policies:
   1. Vacation Policy (score: 8.724)
   2. Remote Work Policy (score: 4.924)
   3. Health Insurance (score: 4.235)

❓ Question: Can I work from home?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

💡 Answer:
Certainly! Yes, you can work from home up to 3 days per week, as outlined in TechCorp's **Remote Work Policy**. 

In [16]:
def generate_python_docs() -> List[Document]:
    """
    Generate sample Python documentation snippets.
    """
    python_docs = [
        {
            "title": "List Comprehensions",
            "content": """List comprehensions provide a concise way to create lists. 
            Common applications are to make new lists where each element is the result of some 
            operations applied to each member of another sequence. 
            Syntax: [expression for item in iterable if condition]
            Example: squares = [x**2 for x in range(10)]""",
            "topic": "data_structures"
        },
        {
            "title": "Dictionaries",
            "content": """Dictionaries are unordered collections of key-value pairs. 
            Keys must be immutable (strings, numbers, tuples). Values can be any type.
            Creation: my_dict = {'key': 'value', 'number': 42}
            Access: my_dict['key'] or my_dict.get('key')
            Methods: keys(), values(), items(), update(), pop()""",
            "topic": "data_structures"
        },
        {
            "title": "Exception Handling",
            "content": """Use try-except blocks to handle exceptions gracefully.
            Syntax: try: risky_operation() except ExceptionType as e: handle_error(e)
            Multiple exceptions: except (TypeError, ValueError) as e:
            Finally block: Executes regardless of exception. Use for cleanup.
            Raise custom exceptions: raise ValueError('Invalid input')""",
            "topic": "error_handling"
        },
        {
            "title": "File I/O",
            "content": """Python provides built-in functions for file operations.
            Open file: with open('file.txt', 'r') as f: content = f.read()
            Modes: 'r' (read), 'w' (write), 'a' (append), 'rb' (read binary)
            Methods: read(), readline(), readlines(), write(), writelines()
            Context manager (with) automatically closes files.""",
            "topic": "io"
        },
        {
            "title": "Lambda Functions",
            "content": """Lambda functions are small anonymous functions defined with lambda keyword.
            Syntax: lambda arguments: expression
            Example: square = lambda x: x**2
            Common with map, filter, sorted: sorted(items, key=lambda x: x['value'])
            Limited to single expression, no statements.""",
            "topic": "functions"
        },
        {
            "title": "Generators",
            "content": """Generators are functions that return iterators using yield.
            Memory efficient for large sequences - values generated on-the-fly.
            Example: def count_up_to(n): for i in range(n): yield i
            Generator expressions: (x**2 for x in range(10))
            Use next() to get values or iterate in for loop.""",
            "topic": "iterators"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={"title": doc["title"], "topic": doc["topic"], "source": "python_docs"}
        )
        for doc in python_docs
    ]

# Generate Python documentation
python_docs = generate_python_docs()
print(f"✅ Generated {len(python_docs)} Python documentation snippets")

✅ Generated 6 Python documentation snippets


In [18]:
python_store = InMemoryDocumentStore()
python_store.write_documents(python_docs)

code_template = """
You are a Python programming assistant. Help the user understand Python concepts based on the documentation.

Documentation:
{% for doc in documents %}
## {{ doc.meta.title }}
{{ doc.content }}

{% endfor %}

Question: {{ query }}

Provide a clear explanation with code examples where applicable.
Answer:
"""

llm = AzureOpenAIGenerator(
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version="2024-12-01-preview",
    generation_kwargs={"temperature": 0.3},
)
python_pipeline = Pipeline()
python_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=python_store))
python_pipeline.add_component("prompt_builder", PromptBuilder(template=code_template))
python_pipeline.add_component("llm",llm)

python_pipeline.connect("retriever.documents", "prompt_builder.documents")
python_pipeline.connect("prompt_builder", "llm")

print("✅ Python documentation pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ Python documentation pipeline ready!


In [19]:
python_questions = [
    "How do I handle errors in Python?",
    "What's the difference between a list comprehension and a generator?",
    "How do I read a file safely?"
]

for question in python_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(python_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")


❓ How do I handle errors in Python?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

Hello! Thank you for your question. The company policies provided do not include technical guidance on handling errors in Python or any programming-related issues. However, I recommend reaching out to your team lead, consulting internal technical resources, or collaborating with your peers for assistance. 

If you have any questions about company policies or need further support, feel free to ask. I'm here to help!

Best regards,  
TechCorp HR Assistant

❓ What's the difference between a list comprehension and a generator?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

Thank you for your question! Unfortunately, the difference between a list comprehension and a generator is not covered in TechCorp's company policies, such as the Code of Conduct or Equipment and Technology policies. However, I recommend reaching out to a colleague or team member who speci

In [25]:
def generate_news_articles() -> List[Document]:
    """
    Generate sample news articles.
    """
    news_data = [
        {
            "title": "Major Breakthrough in Quantum Computing",
            "content": """Scientists at Tech University announced a significant breakthrough in 
            quantum computing, achieving quantum supremacy with a 1000-qubit processor. The new 
            system can solve complex optimization problems 1000x faster than classical supercomputers. 
            Lead researcher Dr. Smith states this could revolutionize drug discovery and climate modeling.""",
            "category": "Technology",
            "date": "2025-01-28",
            "source_url": "https://technews.example.com/quantum-breakthrough"
        },
        {
            "title": "New Climate Agreement Reached",
            "content": """World leaders agreed to reduce carbon emissions by 50% by 2035 at the 
            Global Climate Summit. The agreement includes $500 billion in funding for renewable energy 
            in developing nations. Environmental groups praised the commitment while noting challenges 
            in implementation.""",
            "category": "Environment",
            "date": "2025-01-27",
            "source_url": "https://worldnews.example.com/climate-agreement"
        },
        {
            "title": "AI Assists in Medical Diagnosis",
            "content": """A new AI system developed by MedTech Corp can diagnose rare diseases 
            with 95% accuracy from medical imaging. The system has been trained on 10 million medical 
            images and can identify patterns invisible to human doctors. Clinical trials show 30% 
            improvement in early detection rates.""",
            "category": "Healthcare",
            "date": "2025-01-26",
            "source_url": "https://healthnews.example.com/ai-diagnosis"
        },
        {
            "title": "Electric Vehicle Sales Surge",
            "content": """Electric vehicle sales increased 200% year-over-year, now comprising 
            25% of all new car sales globally. Major automakers announced plans to phase out combustion 
            engines by 2030. Battery technology improvements have extended range to 500+ miles while 
            reducing costs by 40%.""",
            "category": "Automotive",
            "date": "2025-01-25",
            "source_url": "https://autonews.example.com/ev-sales-surge"
        }
    ]
    
    return [
        Document(
            content=doc["content"],
            meta={
                "title": doc["title"],
                "category": doc["category"],
                "date": doc["date"],
                "source_url": doc["source_url"],
                "source": "news"
            }
        )
        for doc in news_data
    ]

news_docs = generate_news_articles()
print(f"✅ Generated {len(news_docs)} news articles")

✅ Generated 4 news articles


In [26]:
news_template = """
You are a news assistant. Answer questions based on the news articles provided.

News Articles:
{% for doc in documents %}
---
Title: {{ doc.meta.title }}
Date: {{ doc.meta.date }}
Source: {{ doc.meta.source_url }}
Content: {{ doc.content }}
{% endfor %}

Question: {{ query }}

Instructions:
- Answer based on the articles provided
- Include citations with article titles and dates
- If information is not in the articles, state that clearly
- Provide source URLs when relevant

Answer:
"""

# Build news pipeline
news_store = InMemoryDocumentStore()
news_store.write_documents(news_docs)

llm = AzureOpenAIGenerator(
    api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    api_version="2024-12-01-preview",
    generation_kwargs={"temperature": 0.3},
)

news_pipeline = Pipeline()
news_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=news_store))
news_pipeline.add_component("prompt_builder", PromptBuilder(template=news_template))
news_pipeline.add_component("llm", llm)

news_pipeline.connect("retriever.documents", "prompt_builder.documents")
news_pipeline.connect("prompt_builder", "llm")

print("✅ News pipeline ready!")

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


✅ News pipeline ready!


In [27]:
news_questions = [
    "What are the recent developments in AI for healthcare?",
    "What progress has been made in quantum computing?",
    "What's happening with electric vehicles?"
]

for question in news_questions:
    print("\n" + "="*80)
    print(f"❓ {question}")
    print("="*80)
    
    result = ask_question(news_pipeline, question, top_k=2)
    print(f"\n{result['answer']}")
    
    print("\n📰 Sources:")
    for doc in result['retrieved_docs']:
        print(f"  • {doc.meta['title']} ({doc.meta['date']})")
        print(f"    {doc.meta['source_url']}")


❓ What are the recent developments in AI for healthcare?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

Recent developments in AI for healthcare include the creation of a new AI system by MedTech Corp that can diagnose rare diseases with 95% accuracy using medical imaging. The system has been trained on 10 million medical images and is capable of identifying patterns that are invisible to human doctors. Clinical trials have demonstrated a 30% improvement in early detection rates. 

(Source: "AI Assists in Medical Diagnosis," 2025-01-26, https://healthnews.example.com/ai-diagnosis)

📰 Sources:
  • New Climate Agreement Reached (2025-01-27)
    https://worldnews.example.com/climate-agreement
  • AI Assists in Medical Diagnosis (2025-01-26)
    https://healthnews.example.com/ai-diagnosis

❓ What progress has been made in quantum computing?
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

A major breakthrough in quantum computing has been ac

In [28]:
# Different prompt templates to compare
prompt_variations = {
    "basic": """
Context: {% for doc in documents %}{{ doc.content }}{% endfor %}
Question: {{ query }}
Answer:
""",
    
    "structured": """
You are a helpful assistant. Use the context below to answer the question.

CONTEXT:
{% for doc in documents %}
[{{ loop.index }}] {{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

ANSWER:
""",
    
    "with_instructions": """
You are a precise assistant. Follow these guidelines:
1. Base your answer ONLY on the context provided
2. If the context doesn't contain the answer, say "I don't have enough information"
3. Be concise but complete
4. Cite specific parts of the context when possible

CONTEXT:
{% for doc in documents %}
Source {{ loop.index }}: {{ doc.content }}
{% endfor %}

USER QUESTION: {{ query }}

YOUR ANSWER:
""",
    
    "chain_of_thought": """
You are a thoughtful assistant. Answer the question step-by-step.

CONTEXT:
{% for doc in documents %}
{{ doc.content }}
{% endfor %}

QUESTION: {{ query }}

Let's think through this step by step:
1. First, identify relevant information from the context
2. Then, synthesize the information
3. Finally, provide a clear answer

ANSWER:
"""
}

print("✅ Created 4 prompt variations for comparison")

✅ Created 4 prompt variations for comparison


In [29]:
def compare_prompt(question: str, store: InMemoryDocumentStore, templates= Dict[str, str]):
    """
    Compare different prompt templates for the same question.
    """
    print(f"\n{'='*80}")
    print(f"Comparing prompts for: {question}")
    print(f"{'='*80}\n")
    

    for name, template in templates.items():
        llm = AzureOpenAIGenerator(
            api_key=Secret.from_env_var("AZURE_OPENAI_API_KEY"),
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
            api_version="2024-12-01-preview",
            generation_kwargs={"temperature": 0.3},
        )

        temp_pipeline = Pipeline()
        temp_pipeline.add_component("retriever", InMemoryBM25Retriever(document_store=store))
        temp_pipeline.add_component("prompt_builder", PromptBuilder(template=template))
        temp_pipeline.add_component("llm",llm)

        temp_pipeline.connect("retriever.documents", "prompt_builder.documents")
        temp_pipeline.connect("prompt_builder", "llm")

        result = ask_question(temp_pipeline, question, top_k=2)
        
        print(f"\n📝 Template: {name.upper()}")
        print("-" * 80)
        print(result['answer'])
        print(f"\n⏱️ Response time: {result['response_time']:.2f}s")

compare_prompt(
    "How many vacation days do employees get?",
    company_store,
    prompt_variations
)

PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.



Comparing prompts for: How many vacation days do employees get?



PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: BASIC
--------------------------------------------------------------------------------
Full-time employees receive **20 days of paid vacation annually**, accruing at a rate of **1.67 days per month**.

⏱️ Response time: 2.39s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: STRUCTURED
--------------------------------------------------------------------------------
Employees receive **20 days of paid vacation annually**.

⏱️ Response time: 2.04s


PromptBuilder has 2 prompt variables, but `required_variables` is not set. By default, all prompt variables are treated as optional, which may lead to unintended behavior in multi-branch pipelines. To avoid unexpected execution, ensure that variables intended to be required are explicitly set in `required_variables`.


DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: WITH_INSTRUCTIONS
--------------------------------------------------------------------------------
Employees receive 20 days of paid vacation annually, accruing at a rate of 1.67 days per month. (Source 1)

⏱️ Response time: 2.81s
DEBUG - Result keys: dict_keys(['retriever', 'prompt_builder', 'llm'])

📝 Template: CHAIN_OF_THOUGHT
--------------------------------------------------------------------------------
### Step 1: Identify relevant information from the context
- Full-time employees receive **20 days of paid vacation annually**.
- Vacation days **accrue monthly** at a rate of **1.67 days per month**.
- Unused vacation days can be **carried over up to 5 days into the next year**.

### Step 2: Synthesize the information
- The annual vacation allowance is **20 days** for full-time employees.
- These 20 days are earned gradually over the year at a rate of **1.67 days per month**.
- If an employee does

### 8. Basic Evaluation

In [ ]:
def evaluate_retrieval_precision(pipeline: Pipeline, test_cases: List[Dict[str, Any]]):
    """
    Evaluate retrieval precision - are retrieved documents relevant?
    
    Args:
        pipeline: RAG pipeline to evaluate
        test_cases: List of dicts with 'query' and 'expected_doc_titles'
    """
    results = []

    for case in test_cases:
        query = case['query']
        expected_titles = set(case'expected_doc_titles'])